# CelebA STAQ

A notebook for the canonical CelebA STAQ workflow.

Main choices in this path:

- prediction target: `Smiling`
- query vocabulary: CelebA attributes excluding `Smiling`
- sensitive set: a small gender-presentation subset
- Concept-QA supervision: direct per-image CelebA attributes
- analysis: replay examples first, then one minimal fixed-history comparison

If CelebA is not already present under `data/`, set `download_celeba = True` in the setup cell once.

In [ ]:
%pip install -e ..

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from staq.analysis import (
    evaluate_bundles_on_fixed_histories,
    plot_rollout_comparisons,
    probe_topk_sensitive_queries,
    sample_intuition_replays,
)
from staq.config import CelebAStaqConfig, default_paths
from staq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from staq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    get_celeba_loaders,
    get_raw_celeba_dataset,
    load_celeba_attribute_spec,
)
from staq.models import ConceptNet2
from staq.sensitive_labels import (
    build_sensitive_labels_from_concept_targets,
    load_sensitive_labels,
    save_sensitive_labels,
)
from staq.training import HistorySamplingConfig, build_staq_models, fit_concept_qa, fit_staq, seed_everything

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = CelebAStaqConfig()
device = config.device
seed_everything(config.random_seed)

figure_ext = ".svg"
download_celeba = False
experiment_name = "celeba_smiling_filtered_shortcuts_rs8"
replay_max_steps = 1

train_min_history = 0
train_max_history = 8
train_non_sensitive_only = False

replay_min_history = 1
replay_max_history = 2
replay_history_mode = "non_sensitive"

concept_qa_max_train_batches = 80 if device.type == "cpu" else None
concept_qa_max_eval_batches = 20 if device.type == "cpu" else None
staq_max_train_batches = 80 if device.type == "cpu" else None
staq_max_eval_batches = 20 if device.type == "cpu" else None
probe_pool_size = 2000 if device.type == "cpu" else 10000


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")
print(f"experiment_name: {experiment_name}")
print(f"train history: {train_min_history}..{train_max_history} | non_sensitive_only={train_non_sensitive_only}")
print(f"replay history: {replay_min_history}..{replay_max_history} | mode={replay_history_mode}")
print(f"replay_max_steps: {replay_max_steps}")

In [ ]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    excluded_query_attributes=config.excluded_query_attributes,
    download=download_celeba,
)
positive_class_idx = 1
positive_class_name = spec.class_names[positive_class_idx]
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)

qa_train_loader, qa_valid_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=download_celeba,
)
staq_train_loader, staq_valid_loader, _ = get_celeba_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    return_query_targets=True,
    download=download_celeba,
)
_, _, test_ds = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=False,
    download=download_celeba,
)
raw_test_ds = get_raw_celeba_dataset(paths.data_root, spec=spec, split="test", download=False)

print(f"target attribute: {spec.target_attribute}")
print(f"# queries: {len(concepts)}")
print(f"excluded shortcut attributes: {config.excluded_query_attributes}")
print(f"sensitive attributes: {spec.sensitive_attribute_names}")
print(f"positive class for plots: {positive_class_name}")
print(f"train/valid/test sizes: {len(staq_train_loader.dataset)}, {len(staq_valid_loader.dataset)}, {len(test_ds)}")

In [ ]:
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{experiment_name}_history.json"

if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
    qa_source = qa_checkpoint
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        qa_optimizer,
        T_max=max(config.concept_qa_epochs, 1),
    )
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_valid_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=config.concept_qa_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=None,
        clip_device=device,
        train_device=device,
        max_train_batches=concept_qa_max_train_batches,
        max_eval_batches=concept_qa_max_eval_batches,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(qa_history_path, "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
    qa_source = qa_checkpoint

print(f"Concept-QA ready from: {qa_source}")

In [ ]:
sensitive_labels_dir = paths.artifacts_root / "sensitive_labels" / experiment_name

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels_from_concept_targets(
        concept_targets=staq_train_loader.dataset.query_targets,
        sens_idx=sens_idx,
    )
    s_soft_test, s_hard_test = build_sensitive_labels_from_concept_targets(
        concept_targets=test_ds.query_targets,
        sens_idx=sens_idx,
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Mean sensitive activation (train/test):",
    float(sensitive_label_cache["s_soft_train"].mean()),
    float(sensitive_label_cache["s_soft_test"].mean()),
)

In [ ]:
def load_or_train_bundle(
    run_name,
    lambda_adv,
    alpha_sens,
    min_history=train_min_history,
    max_history=train_max_history,
    non_sensitive_only=train_non_sensitive_only,
    epochs=config.default_train_epochs,
    learning_rate=config.learning_rate,
    max_train_batches=staq_max_train_batches,
    max_eval_batches=staq_max_eval_batches,
    force_retrain=False,
):
    ckpt_path = paths.checkpoints_root / f"{experiment_name}_{run_name}_best.pt"
    history_path = paths.runs_root / f"{experiment_name}_{run_name}_history.json"
    if ckpt_path.exists() and not force_retrain:
        return load_run_bundle(
            ckpt_path,
            device=device,
            max_queries=len(concepts),
            num_classes=config.num_classes,
            actor_eps=config.actor_eps,
        )

    actor, classifier, s_head = build_staq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_staq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=staq_train_loader,
        test_loader=staq_valid_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_adv=lambda_adv,
        alpha_sens=alpha_sens,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_eval_batches,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_adv": lambda_adv,
            "alpha_sens": alpha_sens,
            "target_attribute": spec.target_attribute,
            "sensitive_attributes": spec.sensitive_attribute_names,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_run_bundle(
        ckpt_path,
        device=device,
        max_queries=len(concepts),
        num_classes=config.num_classes,
        actor_eps=config.actor_eps,
    )


baseline_bundle = load_or_train_bundle("baseline", lambda_adv=0.0, alpha_sens=0.0)
staq_bundle = load_or_train_bundle("lam_0.40", lambda_adv=0.4, alpha_sens=0.0)

print(baseline_bundle["ckpt_path"])
print(staq_bundle["ckpt_path"])


def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

In [ ]:
baseline_first_query_probe = probe_topk_sensitive_queries(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundle=baseline_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    matched_sensitive_concepts=spec.sensitive_attribute_names,
    pool_size=probe_pool_size,
    num_trials=3,
    min_history=replay_min_history,
    max_history=replay_max_history,
    history_mode=replay_history_mode,
    topk_steps=1,
    random_seed=config.random_seed,
    label_tag="S",
)
staq_first_query_probe = probe_topk_sensitive_queries(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    matched_sensitive_concepts=spec.sensitive_attribute_names,
    pool_size=probe_pool_size,
    num_trials=3,
    min_history=replay_min_history,
    max_history=replay_max_history,
    history_mode=replay_history_mode,
    topk_steps=1,
    random_seed=config.random_seed,
    label_tag="S",
)

first_query_probe_path = paths.runs_root / f"{experiment_name}_first_query_probe.json"
with open(first_query_probe_path, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "baseline": baseline_first_query_probe,
            "lam_0.40": staq_first_query_probe,
        },
        handle,
        indent=2,
    )

{
    "baseline_top_first_queries": baseline_first_query_probe["summary"]["top_overall_concepts_in_topk"][:10],
    "baseline_top_sensitive_first_queries": baseline_first_query_probe["summary"]["top_sensitive_concepts"][:10],
    "lam_0.40_top_first_queries": staq_first_query_probe["summary"]["top_overall_concepts_in_topk"][:10],
    "lam_0.40_top_sensitive_first_queries": staq_first_query_probe["summary"]["top_sensitive_concepts"][:10],
    "saved_probe": str(first_query_probe_path),
}

In [ ]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    num_cases=4,
    pool_size=500 if device.type == "cpu" else 1500,
    num_trials=3,
    min_history=replay_min_history,
    max_history=replay_max_history,
    history_mode=replay_history_mode,
    prefer_baseline_sensitive=False,
    confidence_threshold=config.confidence_threshold,
    rollout_max_steps=replay_max_steps,
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path(f"{experiment_name}_one_step_replay_examples"),
    title_prefix="one-step replay",
)

print(f"Saved one-step replay figure: {intuition_fig}")

## Optional Comparison

Keep the aggregate comparison minimal.
Use this section only to compare `baseline` vs `lam=0.4` on fixed histories.
The main CelebA story should still come from replay behavior.

In [ ]:
comparison_name = f"{experiment_name}_baseline_vs_lam_0.4"
comparison_num_trials = 8
comparison_history_mode = replay_history_mode
comparison_max_samples = 2000 if device.type == "cpu" else None

comparison_bundles = {
    "baseline": baseline_bundle,
    "lam=0.4": staq_bundle,
}

print("Using current main runs for CelebA comparison:")
print("- baseline ->", baseline_bundle["ckpt_path"])
print("- lam=0.4 ->", staq_bundle["ckpt_path"])

In [ ]:
comparison_fixed_history_rows = evaluate_bundles_on_fixed_histories(
    dataset=test_ds,
    answer_builder=answer_builder,
    bundles_by_name=comparison_bundles,
    sensitive_mask=sensitive_mask,
    min_history=replay_min_history,
    max_history=replay_max_history,
    history_mode=comparison_history_mode,
    num_trials=comparison_num_trials,
    max_samples=comparison_max_samples,
    eval_seed=config.random_seed,
    desc="Fixed-history comparison",
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
)

comparison_fixed_history_path = paths.runs_root / f"{comparison_name}_fixed_history_summary.json"
with open(comparison_fixed_history_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_fixed_history_rows, handle, indent=2)

baseline_row = next(row for row in comparison_fixed_history_rows if row["run_name"] == "baseline")
baseline_acc = baseline_row["mean_acc"]
baseline_sens = baseline_row["mean_sensitive_query_rate"]

comparison_table = [
    {
        "run_name": row["run_name"],
        "lambda_adv": row["lambda_adv"],
        "mean_acc": round(row["mean_acc"], 4),
        "acc_delta_vs_baseline": round(row["mean_acc"] - baseline_acc, 4),
        "std_acc": round(row["std_acc"], 4),
        "mean_sensitive_query_rate": round(row["mean_sensitive_query_rate"], 4),
        "sens_delta_vs_baseline": round(row["mean_sensitive_query_rate"] - baseline_sens, 4),
        "std_sensitive_query_rate": round(row["std_sensitive_query_rate"], 4),
        "mean_smiling_prob": round(row["mean_positive_prob"], 4),
    }
    for row in comparison_fixed_history_rows
]

comparison_table_path = paths.runs_root / f"{comparison_name}_summary_table.json"
with open(comparison_table_path, "w", encoding="utf-8") as handle:
    json.dump(comparison_table, handle, indent=2)

print(f"Saved fixed-history summary: {comparison_fixed_history_path}")
print(f"Saved comparison table: {comparison_table_path}")

comparison_table